In [1]:
from langchain_core.documents import Document

In [2]:
doc=Document(
    page_content="this is the main text content I am using to create RAG",
    metadata={
        "source":"exmaple.txt",
        "pages":1,
        "author":"Shashank MN",
        "date_created":"2026-01-01"
    }
)
doc

Document(metadata={'source': 'exmaple.txt', 'pages': 1, 'author': 'Shashank MN', 'date_created': '2026-01-01'}, page_content='this is the main text content I am using to create RAG')

In [3]:
import os
os.makedirs("../data/text_files", exist_ok=True)


In [4]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [9]:
from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [10]:
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [11]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Pattern to match files  
    loader_cls= PyMuPDFLoader, ##loader class to use
    show_progress=False

)

pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'PDPreStamp v3.3', 'creator': 'Adobe InDesign 16.4 (Windows)', 'creationdate': '2022-09-29T17:54:00+02:00', 'source': '..\\data\\pdf\\iso27001.pdf', 'file_path': '..\\data\\pdf\\iso27001.pdf', 'total_pages': 26, 'format': 'PDF 1.4', 'title': 'ISO/IEC 27001:2022', 'author': 'ISO', 'subject': '', 'keywords': '', 'moddate': '2022-10-26T19:22:40+08:00', 'trapped': '', 'modDate': "D:20221026192240+08'00'", 'creationDate': "D:20220929175400+02'00'", 'page': 0}, page_content="Information security, cybersecurity \nand privacy protection\xa0— Information \nsecurity management systems — \nRequirements\nSécurité de l'information, cybersécurité et protection de la vie \nprivée\xa0— Systèmes de management de la sécurité de l'information — \nExigences\nINTERNATIONAL \nSTANDARD\nISO/IEC \n27001\nThird edition \n2022-10\nReference number \nISO/IEC 27001:2022(E)\n© ISO/IEC 2022\n--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`---"),
 Document(metadata={'producer': 'PDPreSt

In [12]:
type(pdf_documents[0])

langchain_core.documents.base.Document

In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\HP\OneDrive\Desktop\Rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

In [15]:
embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


c:\Users\HP\OneDrive\Desktop\Rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\HP\OneDrive\Desktop\Rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149:

Model loaded successfully. Embedding dimension: 384


C:\Users\HP\AppData\Local\Temp\ipykernel_4460\2492392461.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [16]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [19]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [46]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from pathlib import Path

In [47]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: iso27001.pdf
  ✓ Loaded 26 pages

Total documents loaded: 26


In [48]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [50]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [51]:
chunks=split_documents(all_pdf_documents)
chunks

Split 26 documents into 76 chunks

Example chunk:
Content: Information security, cybersecurity 
and privacy protection — Information 
security management systems — 
Requirements
Sécurité de l'information, cybersécurité et protection de la vie 
privée — Systèm...
Metadata: {'producer': 'PDPreStamp v3.3', 'creator': 'Adobe InDesign 16.4 (Windows)', 'creationdate': '2022-09-29T17:54:00+02:00', 'author': 'ISO', 'license': 'Information Handling Services, 2022', 'moddate': '2022-10-26T19:22:40+08:00', 'title': 'ISO/IEC 27001:2022', 'trapped': '/False', 'source': '..\\data\\pdf\\iso27001.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'source_file': 'iso27001.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PDPreStamp v3.3', 'creator': 'Adobe InDesign 16.4 (Windows)', 'creationdate': '2022-09-29T17:54:00+02:00', 'author': 'ISO', 'license': 'Information Handling Services, 2022', 'moddate': '2022-10-26T19:22:40+08:00', 'title': 'ISO/IEC 27001:2022', 'trapped': '/False', 'source': '..\\data\\pdf\\iso27001.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1', 'source_file': 'iso27001.pdf', 'file_type': 'pdf'}, page_content="Information security, cybersecurity \nand privacy protection — Information \nsecurity management systems — \nRequirements\nSécurité de l'information, cybersécurité et protection de la vie \nprivée — Systèmes de management de la sécurité de l'information — \nExigences\nINTERNATIONAL \nSTANDARD\nISO/IEC \n27001\nThird edition  \n2022-10\nReference number \nISO/IEC 27001:2022(E)\n© ISO/IEC 2022\n--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`---"),
 Document(metadata={'producer': 'PDPreStamp v3.3', 'creator': 'Adobe InDesign 16.4 (Windows)', '

In [52]:
texts=[doc.page_content for doc in chunks]
texts

["Information security, cybersecurity \nand privacy protection — Information \nsecurity management systems — \nRequirements\nSécurité de l'information, cybersécurité et protection de la vie \nprivée — Systèmes de management de la sécurité de l'information — \nExigences\nINTERNATIONAL \nSTANDARD\nISO/IEC \n27001\nThird edition  \n2022-10\nReference number \nISO/IEC 27001:2022(E)\n© ISO/IEC 2022\n--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`---",
 'ii\nISO/IEC 27001:2022(E)\nCOPYRIGHT PROTECTED DOCUMENT\n©  ISO/IEC 2022\nAll rights reserved. Unless otherwise specified, or required in the context of its implementation, no part of this publication may \nbe reproduced or utilized otherwise in any form or by any means, electronic or mechanical, including photocopying, or posting on \nthe internet or an intranet, without prior written permission. Permission can be requested from either ISO at the address below \nor ISO’s member body in the country of the requester.\nISO copyright office\nCP 

In [53]:
embeddings=embedding_manager.generate_embeddings(texts)
embeddings

Generating embeddings for 76 texts...


Batches: 100%|██████████| 3/3 [00:01<00:00,  1.59it/s]

Generated embeddings with shape: (76, 384)


array([[-0.02289787, -0.00367537, -0.0773754 , ...,  0.00134361,
         0.03371682,  0.02073073],
       [-0.09013383,  0.03090609, -0.00227919, ..., -0.04632635,
        -0.00143569, -0.00497085],
       [-0.00727987, -0.00416383, -0.04750561, ...,  0.0377786 ,
         0.02052876, -0.0190212 ],
       ...,
       [-0.00506967,  0.04254375, -0.0481151 , ...,  0.00177925,
         0.04341702, -0.01447488],
       [-0.00423194,  0.0218449 , -0.06934411, ..., -0.10534475,
        -0.03728397,  0.0335189 ],
       [-0.03979393,  0.02946173, -0.05601009, ..., -0.02299315,
        -0.03232608, -0.02320939]], shape=(76, 384), dtype=float32)

In [55]:
vectorstore.add_documents(chunks,embeddings)

Adding 76 documents to vector store...
Successfully added 76 documents to vector store
Total documents in collection: 76


In [56]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [60]:
rag_retriever.retrieve(" what is information security, cybersecurity and privacy protection")

Retrieving documents for query: ' what is information security, cybersecurity and privacy protection'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 142.74it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_76156f89_0',
  'content': "Information security, cybersecurity \nand privacy protection — Information \nsecurity management systems — \nRequirements\nSécurité de l'information, cybersécurité et protection de la vie \nprivée — Systèmes de management de la sécurité de l'information — \nExigences\nINTERNATIONAL \nSTANDARD\nISO/IEC \n27001\nThird edition  \n2022-10\nReference number \nISO/IEC 27001:2022(E)\n© ISO/IEC 2022\n--``,,,,,``````,,,,,`,`,`,`,,`,-`-`,,`,,`,`,,`---",
  'metadata': {'page_label': '1',
   'doc_index': 0,
   'moddate': '2022-10-26T19:22:40+08:00',
   'source_file': 'iso27001.pdf',
   'file_type': 'pdf',
   'creationdate': '2022-09-29T17:54:00+02:00',
   'author': 'ISO',
   'total_pages': 26,
   'source': '..\\data\\pdf\\iso27001.pdf',
   'producer': 'PDPreStamp v3.3',
   'trapped': '/False',
   'creator': 'Adobe InDesign 16.4 (Windows)',
   'page': 0,
   'title': 'ISO/IEC 27001:2022',
   'license': 'Information Handling Services, 2022',
   'content_length'

In [61]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file


True

In [62]:
groq_api_key = os.getenv("GROQ_API_KEY")

In [76]:
llm=ChatGroq(
    api_key=groq_api_key,model="openai/gpt-oss-20b",temperature=0.7,max_tokens=1024)

In [77]:
class GroqLLM:
    def __init__(self, model_name: str = "openai/gpt-oss-20b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [78]:
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [79]:
answer=rag_simple(" Information security, cybersecurity and privacy protection",rag_retriever,llm)
print(answer)

Retrieving documents for query: ' Information security, cybersecurity and privacy protection'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 176.27it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


ISO/IEC 27001:2022 requires an organization to implement an integrated Information Security Management System (ISMS) that:

- **Manages risks** and scales to the organization’s needs.  
- **Integrates security** into all processes, information systems, and controls.  
- **Segregates duties** to avoid conflicting responsibilities.  
- **Mandates management responsibility** for enforcing security policies and procedures.  
- **Maintains contacts** with authorities, special‑interest groups, and security forums.  
- **Collects and analyses threat intelligence** to inform security decisions.  
- **Incorporates security into project management** throughout the project lifecycle.


In [81]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Information security, cybersecurity and privacy protection", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Information security, cybersecurity and privacy protection'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 105.62it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: **Information security, cybersecurity and privacy protection**  
ISO/IEC 27001:2022 defines the requirements for an Information Security Management System (ISMS) that integrates security into all organisational processes, ensures segregation of duties, management responsibility, contact with authorities and interest groups, threat intelligence, and security‑aware project management, thereby safeguarding confidentiality, integrity, availability and privacy of information.
Sources: [{'source': 'iso27001.pdf', 'page': 0, 'score': 0.2699015140533447, 'preview': "Information security, cybersecurity \nand privacy protection — Information \nsecurity management systems — \nRequirements\nSécurité de l'information, cybersécurité et protection de la vie \nprivée — Systèmes de management de la sécurité de l'information — \nExigences\nINTERNATIONAL \nSTANDARD\nISO/IEC \n27001..."}, {'source': 'iso27001.pdf', 'page': 4, 'score': 0.15816032886505127, 'preview': 'risks are adequately managed.\